# NSMC 감성 분류 (간소화 버전)

네이버 영화 리뷰(NSMC)로 긍정/부정 이진 분류 모델을 학습합니다.

1. Mecab 토큰화 → 단어 사전(상위 10,000개) → 정수 인코딩 → 패딩
2. LSTM / CNN / GlobalPooling 3종 모델 학습 비교 (조기종료 포함)
3. 학습 곡선 시각화, 임베딩 유사 단어 분석
4. 사전학습 Word2Vec 임베딩 vs 무작위 초기화 임베딩 대조 실험

> 원본 대비 변경점: 클래스 3개(DataHandler/TrainHandler/EarlyStopping) → 함수로 통합,
> 사용되지 않는 인코딩/디코딩 헬퍼 제거, 특수 토큰이 전부 빈 문자열이던 버그 수정,
> `np.Inf`(NumPy 2.0 제거됨) → `float('inf')` 교체. 학습 로직과 하이퍼파라미터는 동일합니다.

In [ ]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Mecab
from gensim.models.word2vec import Word2Vec

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

BASE_PATH = os.path.join(os.getcwd(), '..', '..', 'work', 'sentiment_classification')
DATA_PATH = os.path.join(BASE_PATH, 'data')
STOPWORDS = {'의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다'}
PAD, BOS, UNK, UNUSED = 0, 1, 2, 3  # 특수 토큰 인덱스

## 1. 데이터 로드 · 전처리

In [ ]:
def load_and_tokenize(path, tokenizer):
    """txt 로드 → 중복/결측 제거 → 토큰화 + 불용어 제거"""
    df = pd.read_table(path).drop_duplicates(subset=['document']).dropna(how='any')
    tokens = [[w for w in tokenizer.morphs(s) if w not in STOPWORDS] for s in df['document']]
    return tokens, df['label'].to_numpy()


def build_vocab(token_lists, num_words=10000):
    counter = Counter(w for toks in token_lists for w in toks)
    vocab = ['<PAD>', '<BOS>', '<UNK>', '<UNUSED>'] + [w for w, _ in counter.most_common(num_words - 4)]
    word_to_index = {w: i for i, w in enumerate(vocab)}
    index_to_word = {i: w for w, i in word_to_index.items()}
    return word_to_index, index_to_word


def encode(token_lists, word_to_index):
    return [[word_to_index.get(w, UNK) for w in toks] for toks in token_lists]


def pad_and_tensorize(seqs, max_len):
    seqs = [torch.tensor(s[:max_len]) if len(s) else torch.tensor([PAD]) for s in seqs]
    return pad_sequence(seqs, batch_first=True, padding_value=PAD)

In [ ]:
tokenizer = Mecab()
train_tokens, y_train = load_and_tokenize(os.path.join(DATA_PATH, 'ratings_train.txt'), tokenizer)
test_tokens,  y_test  = load_and_tokenize(os.path.join(DATA_PATH, 'ratings_test.txt'), tokenizer)

word_to_index, index_to_word = build_vocab(train_tokens, num_words=10000)
X_train = encode(train_tokens, word_to_index)
X_test  = encode(test_tokens,  word_to_index)

# 문장 길이 분포 → max_len = 평균 + 2*표준편차 (약 95% 커버)
lengths = [len(s) for s in X_train]
max_len = int(np.mean(lengths) + 2 * np.std(lengths))
print(f"문장 길이: 평균 {np.mean(lengths):.1f}, 표준편차 {np.std(lengths):.1f} → max_len = {max_len}")

X_train_pad = pad_and_tensorize(X_train, max_len)
X_test_pad  = pad_and_tensorize(X_test,  max_len)
print(f"Train 텐서: {X_train_pad.shape}, Test 텐서: {X_test_pad.shape}")

In [ ]:
BATCH_SIZE = 64

full_ds = TensorDataset(X_train_pad, torch.tensor(y_train, dtype=torch.float32))
test_ds = TensorDataset(X_test_pad,  torch.tensor(y_test,  dtype=torch.float32))

val_size = int(len(full_ds) * 0.2)
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
print(f"Train {len(train_ds)} / Val {len(val_ds)} / Test {len(test_ds)}")

## 2. 모델 정의 (LSTM / CNN / GlobalPooling)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, (hidden, _) = self.lstm(self.embedding(x))
        return torch.sigmoid(self.fc(hidden[-1])).squeeze(-1)


class CNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, num_filters=128, filter_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, num_filters, fs) for fs in filter_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)

    def forward(self, x):
        emb = self.embedding(x).permute(0, 2, 1)           # [B, E, L]
        pooled = [torch.relu(conv(emb)).max(dim=2)[0] for conv in self.convs]
        return torch.sigmoid(self.fc(self.dropout(torch.cat(pooled, dim=1)))).squeeze(-1)


class GlobalPoolingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        pooled = self.embedding(x).mean(dim=1)             # Global Average Pooling
        return torch.sigmoid(self.fc2(torch.relu(self.fc1(pooled)))).squeeze(-1)

## 3. 학습 루프 (조기종료 포함)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 있으면 학습, 없으면 평가. (평균 loss, 정확도) 반환"""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.enable_grad() if training else torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(inputs)
            correct += ((outputs >= 0.5).float() == labels).sum().item()
            n += len(inputs)
    return total_loss / n, correct / n


def train(model, name, epochs=15, lr=0.001, patience=3):
    """조기종료 + 최적 가중치 복원. history dict 반환"""
    import copy
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_loss, best_state, stall = float('inf'), None, 0

    print(f"\n=== {name} 학습 시작 (epochs={epochs}, lr={lr}) ===")
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, criterion)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        print(f" Epoch {epoch:02d} | Train {tr_loss:.4f}/{tr_acc:.4f} | Val {va_loss:.4f}/{va_acc:.4f}")

        if va_loss < best_loss:
            best_loss, best_state, stall = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            stall += 1
            if stall >= patience:
                print(f" 조기 종료 (epoch {epoch})")
                break

    model.load_state_dict(best_state)
    print(f" 최적 가중치 복원 (Val Loss {best_loss:.4f})")
    return history

## 4. 베이스라인 3종 학습 · 시각화

In [ ]:
VOCAB_SIZE = len(word_to_index)

models = {
    'LSTM': LSTMModel(VOCAB_SIZE).to(device),
    'CNN': CNNModel(VOCAB_SIZE).to(device),
    'GlobalPooling': GlobalPoolingModel(VOCAB_SIZE).to(device),
}
histories = {name: train(model, name) for name, model in models.items()}

In [ ]:
def plot_histories(histories):
    fig, axes = plt.subplots(len(histories), 2, figsize=(14, 4 * len(histories)), squeeze=False)
    for i, (name, h) in enumerate(histories.items()):
        for j, metric in enumerate(('loss', 'acc')):
            ax = axes[i][j]
            ax.plot(h[f'train_{metric}'], label='Train')
            ax.plot(h[f'val_{metric}'], label='Val')
            ax.set_title(f'{name} - {metric.capitalize()}')
            ax.set_xlabel('Epoch')
            ax.legend()
            ax.grid(True)
    plt.tight_layout()
    plt.show()

plot_histories(histories)

## 5. 학습된 임베딩 공간 분석 (유사 단어)

In [ ]:
def similar_words(model, target_word, top_n=10):
    if target_word not in word_to_index:
        print(f"'{target_word}'는 사전에 없습니다.")
        return
    emb = model.embedding.weight.detach().cpu().numpy()
    target = emb[word_to_index[target_word]]
    norms = np.linalg.norm(emb, axis=1) * np.linalg.norm(target)
    norms[norms == 0] = 1e-8
    sims = emb @ target / norms
    print(f"'{target_word}'와 가까운 단어 Top {top_n}:")
    for rank, idx in enumerate(np.argsort(sims)[::-1][1:top_n + 1], 1):
        print(f" {rank:2d}. {index_to_word.get(idx, '<UNK>')} ({sims[idx]:.4f})")

similar_words(models['LSTM'], '끝', top_n=10)

## 6. 사전학습 Word2Vec 임베딩 대조 실험\n\n동일한 LSTM 구조에 `word2vec_ko.model`의 100차원 임베딩을 이식하고 미세조정(lr=0.0002)하여, 무작위 초기화 베이스라인과 비교합니다.

In [ ]:
w2v_path = os.path.join(DATA_PATH, 'word2vec_ko.model')
w2v = Word2Vec.load(w2v_path)

embedding_matrix = np.zeros((VOCAB_SIZE, 100))
hits = 0
for idx, word in index_to_word.items():
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]
        hits += 1
print(f"Word2Vec 매핑: {hits}/{VOCAB_SIZE} 단어")

w2v_lstm = LSTMModel(VOCAB_SIZE).to(device)
w2v_lstm.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
w2v_lstm.embedding.weight.requires_grad = True  # Fine-tuning 허용

baseline_lstm = LSTMModel(VOCAB_SIZE).to(device)
compare_histories = {
    'LSTM (Random)': train(baseline_lstm, 'LSTM (Random)', lr=0.001),
    'LSTM (W2V Fine-tuned)': train(w2v_lstm, 'LSTM (W2V Fine-tuned)', lr=0.0002),
}
plot_histories(compare_histories)

for name, h in compare_histories.items():
    print(f"{name}: 최고 Val Acc = {max(h['val_acc']):.4f}")